# Day 3 Sample Notebook: Building a Cohort

**OHDSI / OMOP Train-the-Trainer, ALS Therapy Development Institute**

Companion to the Day 3 module. You will build a **new-user riluzole** cohort with an inclusion rule
and an exclusion, see the attrition at each step, then characterize the result.

> **Site specific.** This notebook builds a synthetic CDM in memory so it runs in Colab with no credentials and no PHI. To use real data, replace the SQLite connection with your site's governed CDM connection (Databricks, Snowflake, Postgres, BigQuery, and so on). The SQL logic is the same; only the connection changes. Not everyone uses Databricks.

### Objectives
1. Define an entry event (first riluzole exposure).
2. Apply an inclusion rule (>= 365 days prior observation) and an exclusion (no prior edaravone).
3. Read attrition at each step.
4. Characterize the final cohort.

## Step 0: Build the synthetic CDM

In [ ]:
import sqlite3, numpy as np, pandas as pd
np.random.seed(7)

def build_als_cdm(n=250):
    """Build a tiny SYNTHETIC ALS-flavored OMOP CDM in memory. No real data, no PHI."""
    con = sqlite3.connect(":memory:"); cur = con.cursor()
    cur.executescript("""
    CREATE TABLE concept(concept_id INT, concept_name TEXT, domain_id TEXT, vocabulary_id TEXT,
        concept_class_id TEXT, standard_concept TEXT, concept_code TEXT);
    CREATE TABLE concept_ancestor(ancestor_concept_id INT, descendant_concept_id INT, min_levels_of_separation INT);
    CREATE TABLE person(person_id INT, gender_concept_id INT, year_of_birth INT);
    CREATE TABLE observation_period(person_id INT, observation_period_start_date TEXT, observation_period_end_date TEXT);
    CREATE TABLE condition_occurrence(person_id INT, condition_concept_id INT, condition_start_date TEXT);
    CREATE TABLE drug_exposure(person_id INT, drug_concept_id INT, drug_exposure_start_date TEXT);
    CREATE TABLE measurement(person_id INT, measurement_concept_id INT, measurement_date TEXT, value_as_number REAL);
    CREATE TABLE procedure_occurrence(person_id INT, procedure_concept_id INT, procedure_date TEXT);
    """)
    concepts=[
     (35748069,"Amyotrophic lateral sclerosis","Condition","SNOMED","Clinical Finding","S","86044005"),
     (4134454,"Bulbar onset","Observation","SNOMED","Clinical Finding","S","230258005"),
     (1314865,"riluzole","Drug","RxNorm","Ingredient","S","9468"),
     (19077572,"riluzole 50 MG Oral Tablet","Drug","RxNorm","Clinical Drug","S","316158"),
     (1326727,"edaravone","Drug","RxNorm","Ingredient","S","1546020"),
     (40234834,"edaravone 30 MG/100mL","Drug","RxNorm","Clinical Drug","S","1546022"),
     (9990001,"sodium phenylbutyrate / taurursodiol","Drug","RxNorm","Ingredient","S","2630918"),
     (9990002,"tofersen","Drug","RxNorm","Ingredient","S","2629000"),
     (1304643,"baclofen","Drug","RxNorm","Ingredient","S","1292"),
     (4052536,"Gastrostomy","Procedure","SNOMED","Procedure","S","36245001"),
     (3024171,"Forced vital capacity","Measurement","LOINC","Clinical Obs","S","19868-9"),
     (8532,"FEMALE","Gender","Gender","Gender","S","F"),(8507,"MALE","Gender","Gender","Gender","S","M"),
     (0,"No matching concept","Metadata","None","Undefined",None,"0")]
    cur.executemany("INSERT INTO concept VALUES(?,?,?,?,?,?,?)",concepts)
    cur.executemany("INSERT INTO concept_ancestor VALUES(?,?,?)",[
     (1314865,19077572,1),(1314865,1314865,0),(1326727,40234834,1),(1326727,1326727,0),
     (9990001,9990001,0),(9990002,9990002,0),(1304643,1304643,0)])
    persons=[];obsp=[];cond=[];drug=[];meas=[];proc=[]
    for pid in range(1,n+1):
        sex=int(np.random.choice([8532,8507])); yob=int(np.random.normal(1958,11))
        persons.append((pid,sex,yob))
        dx_year=np.random.randint(2018,2024); dx=f"{dx_year}-{np.random.randint(1,13):02d}-15"
        cond.append((pid,35748069,dx))
        bulbar=np.random.rand()<0.3
        if bulbar: cond.append((pid,4134454,dx))
        prior=int(np.random.choice([400,500,800,200,100],p=[.4,.2,.2,.1,.1]))
        start=pd.Timestamp(dx)-pd.Timedelta(days=prior)
        end=pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(365,1500)))
        obsp.append((pid,start.date().isoformat(),end.date().isoformat()))
        base_fvc=float(np.clip(np.random.normal(85-(10 if bulbar else 0),12),30,110))
        r=np.random.rand()
        if r<0.08: pass                                  # missing (completeness)
        elif r<0.11: meas.append((pid,3024171,dx,float(np.random.choice([0,250.0]))))  # implausible (plausibility)
        else: meas.append((pid,3024171,dx,round(base_fvc,1)))
        idx=pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(0,40)))
        if np.random.rand()<0.85:
            drug.append((pid,19077572,idx.date().isoformat()))
            if np.random.rand()<0.45:
                drug.append((pid,40234834,(idx+pd.Timedelta(days=int(np.random.randint(60,300)))).date().isoformat()))
        else:
            drug.append((pid,40234834,idx.date().isoformat()))
        if np.random.rand()<0.15:
            drug.append((pid,9990001,(idx+pd.Timedelta(days=int(np.random.randint(100,500)))).date().isoformat()))
        if bulbar and np.random.rand()<0.4: drug.append((pid,1304643,idx.date().isoformat()))
        if np.random.rand()<0.05: drug.append((pid,0,idx.date().isoformat()))   # unmapped (conformance)
        age=dx_year-yob
        logit=-2.0+0.04*(age-60)+1.1*bulbar-0.03*(base_fvc-80)
        if np.random.rand()<1/(1+np.exp(-logit)):
            proc.append((pid,4052536,(pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(60,360)))).date().isoformat()))
    cur.executemany("INSERT INTO person VALUES(?,?,?)",persons)
    cur.executemany("INSERT INTO observation_period VALUES(?,?,?)",obsp)
    cur.executemany("INSERT INTO condition_occurrence VALUES(?,?,?)",cond)
    cur.executemany("INSERT INTO drug_exposure VALUES(?,?,?)",drug)
    cur.executemany("INSERT INTO measurement VALUES(?,?,?,?)",meas)
    cur.executemany("INSERT INTO procedure_occurrence VALUES(?,?,?)",proc)
    con.commit(); return con

con = build_als_cdm()
def q(sql): return pd.read_sql_query(sql, con)
print("Synthetic ALS CDM ready:", q("SELECT COUNT(*) n FROM person").n[0], "persons")

## Step 1: Entry event (first riluzole exposure per person)
We use the ingredient concept set (descendants) so any riluzole product qualifies.

In [ ]:
entry = q("""
SELECT de.person_id, MIN(de.drug_exposure_start_date) AS index_date
FROM drug_exposure de
JOIN concept_ancestor ca ON ca.descendant_concept_id = de.drug_concept_id
WHERE ca.ancestor_concept_id = 1314865
GROUP BY de.person_id
""")
print("Step 0 - entry event (any riluzole):", len(entry))
entry.head()

## Step 2: Inclusion rule, prior observation
Keep only people with at least 365 days of observation before their index date. This makes
"new use" meaningful (we could actually see prior history).

In [ ]:
import pandas as pd
op = q("SELECT person_id, observation_period_start_date FROM observation_period")
m = entry.merge(op, on="person_id")
m["prior_days"] = (pd.to_datetime(m.index_date) - pd.to_datetime(m.observation_period_start_date)).dt.days
incl = m[m.prior_days >= 365]
print("Step 1 - with >=365d prior observation:", len(incl))

## Step 3: Exclusion, no prior edaravone
Remove anyone with an edaravone exposure on or before their riluzole index date.

In [ ]:
eda = q("""
SELECT de.person_id, de.drug_exposure_start_date AS eda_date
FROM drug_exposure de
JOIN concept_ancestor ca ON ca.descendant_concept_id = de.drug_concept_id
WHERE ca.ancestor_concept_id = 1326727
""")
prior_eda = incl.merge(eda, on="person_id")
prior_eda = prior_eda[pd.to_datetime(prior_eda.eda_date) <= pd.to_datetime(prior_eda.index_date)]
cohort = incl[~incl.person_id.isin(prior_eda.person_id)]
print("Step 2 - excluding prior edaravone:", len(cohort))

print("\nAttrition")
print(f"  Entry (any riluzole):        {len(entry)}")
print(f"  + >=365d prior observation:  {len(incl)}")
print(f"  + no prior edaravone:        {len(cohort)}")

## Step 4: Characterize the cohort
A first sanity check: who is in it? Age at index, sex, and bulbar-onset fraction.

In [ ]:
feat = q("""
SELECT p.person_id, p.gender_concept_id, p.year_of_birth,
       (SELECT 1 FROM condition_occurrence c WHERE c.person_id=p.person_id AND c.condition_concept_id=4134454) AS bulbar
FROM person p
""").merge(cohort[["person_id","index_date"]], on="person_id")
feat["age_at_index"] = pd.to_datetime(feat.index_date).dt.year - feat.year_of_birth
feat["bulbar"] = feat["bulbar"].fillna(0)
print("Cohort size:", len(feat))
print("Mean age at index:", round(feat.age_at_index.mean(),1))
print("Female %:", round(100*(feat.gender_concept_id==8532).mean(),1))
print("Bulbar onset %:", round(100*feat.bulbar.mean(),1))

## Your turn
1. Loosen the prior-observation window to 180 days. How does the cohort size change?
2. Add a second inclusion rule: index date in 2020 or later.
3. Why is "no prior edaravone" written as a rule that must be FALSE, rather than a separate exclusion list?

<details><summary>Answer key</summary>

1. Change `>= 365` to `>= 180`; the cohort grows because more people clear the lower bar.
2. Filter `cohort` on `pd.to_datetime(index_date).dt.year >= 2020`.
3. OHDSI has no separate exclusion concept. An exclusion is an inclusion criterion whose count of
   qualifying prior events must equal zero. This keeps all logic in one consistent framework.
</details>